In [ ]:
import numpy as np                     # For numerical operations
import pandas as pd                    # For data handling (CSV, tables)
import matplotlib.pyplot as plt        # For plotting graphs

from sklearn.preprocessing import MinMaxScaler   # For scaling data between 0 and 1
from tensorflow.keras.models import Sequential   # To build neural network model
from tensorflow.keras.layers import Dense, SimpleRNN  # Layers used in RNN

# 2. Load Dataset
df = pd.read_csv("Google_Stock_Price.csv", thousands=',')  
# Read CSV file, thousands=',' handles values like 1,000 properly

# Take only 'Open' column (stock opening price)
data = df['Open'].values.reshape(-1, 1)

# Create scaler object (normalize data between 0 and 1)
scaler = MinMaxScaler(feature_range=(0,1))

# Convert data to numeric (remove invalid values like text), drop NaN values
data = pd.to_numeric(df['Open'], errors='coerce').dropna().values.reshape(-1, 1)

# Scale the data
data_scaled = scaler.fit_transform(data)

# Split dataset into training (80%) and testing (20%)
train_size = int(len(data_scaled) * 0.8)

train_data = data_scaled[:train_size]
test_data = data_scaled[train_size:]

# Function to create dataset with time steps (60 previous days → next day prediction)
def create_dataset(dataset):
    X = []   # Input sequences
    y = []   # Output values
    
    # Loop starts from 60 because we need past 60 days data
    for i in range(60, len(dataset)):
        X.append(dataset[i-60:i, 0])  # Last 60 values as input
        y.append(dataset[i, 0])       # Current value as output
    
    return np.array(X), np.array(y)

# Create training and testing datasets
X_train, y_train = create_dataset(train_data)
X_test, y_test = create_dataset(test_data)

# Reshape input to 3D (samples, time steps, features)
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

# Build RNN model
model = Sequential()

# First RNN layer with 50 neurons, returns sequences for next layer
model.add(SimpleRNN(50, return_sequences=True, input_shape=(60,1)))

# Second RNN layer with 50 neurons
model.add(SimpleRNN(50))

# Output layer (1 neuron → predicted price)
model.add(Dense(1))

# Compile model with Adam optimizer and MSE loss
model.compile(optimizer='adam', loss='mean_squared_error')

# Show model structure
model.summary()

# Train the model (20 epochs, batch size = 32)
model.fit(X_train, y_train, epochs=20, batch_size=32)

# Predict on test data
predicted = model.predict(X_test)

# Convert scaled values back to original price
predicted = scaler.inverse_transform(predicted)
real = scaler.inverse_transform(y_test.reshape(-1,1))

# Plot real vs predicted prices
plt.plot(real, color='red', label='Real Price')
plt.plot(predicted, color='blue', label='Predicted Price')

plt.title("Google Stock Price Prediction (RNN)")
plt.xlabel("Time")
plt.ylabel("Price")
plt.legend()
plt.show()

: 